<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Aprendizado de Máquina
Nesta versão da atividade utilizaremos o dataset CIFAR-10.
Características do dataset:
- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor
Importante:
O carregamento do dataset pode ser realizado utilizando:
```python
from tensorflow.keras.datasets import cifar10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```
Após o carregamento:
```python
print(X_train.shape)
```
Saída esperada:
```python
(50000, 32, 32, 3)
```
Onde:
- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.
Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:
```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```
Após o flatten:
```python
print(X_train.shape)
```
Saída esperada:
```python
(50000, 3072)
```
Isso ocorre porque:
```python
32 × 32 × 3 = 3072
```


# Objetivos
Nesta atividade você irá:
- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.
Nesta atividade utilizaremos MLflow para:
- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.


In [1]:
import warnings

warnings.filterwarnings("ignore")


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow


In [3]:
mlflow.set_experiment(
    "assignment"
)


<Experiment: artifact_location='file:///Users/leonardo_mello/atividade-04-deep-learning-i-Leonardo-Mello22/notebooks/mlruns/357366094663847849', creation_time=1779282225298, experiment_id='357366094663847849', last_update_time=1779282225298, lifecycle_stage='active', name='assignment', tags={}>

# Questão 1
Implemente uma função `load_data(seed)` que:
- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:
```python
X_train, X_val, y_train, y_val
```
já normalizados e preparados para treinamento.
Além disso, responda:
1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?


**Solução**:


In [4]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from sklearn.model_selection import train_test_split
from tensorflow.keras.datasets import cifar10

from src.utils import normalize_images, set_seed

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

def load_data(seed, validation_size=0.2, max_samples=6000):
    set_seed(seed)

    (X_train_full, y_train_full), _ = cifar10.load_data()
    y_train_full = y_train_full.ravel()

    original_shape = X_train_full.shape[1:]
    n_features = int(np.prod(original_shape))

    X = X_train_full.reshape(X_train_full.shape[0], -1).astype("float32")
    X = normalize_images(X)
    y = y_train_full

    if max_samples is not None and max_samples < X.shape[0]:
        _, X, _, y = train_test_split(
            X,
            y,
            test_size=max_samples,
            random_state=seed,
            stratify=y,
        )

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=validation_size,
        random_state=seed,
        stratify=y,
    )

    print(f"Formato original das imagens: {original_shape}")
    print(f"Features apos flatten: {n_features}")
    print(f"Treino: {X_train.shape} | Validacao: {X_val.shape}")

    return X_train, X_val, y_train, y_val

SEED = 42
X_train, X_val, y_train, y_val = load_data(SEED)


Formato original das imagens: (32, 32, 3)
Features apos flatten: 3072
Treino: (4800, 3072) | Validacao: (1200, 3072)


### Formato original das imagens?

Cada imagem tem shape (32, 32, 3): 32 pixels de altura, 32 pixels de largura e 3 canais de cor RGB. O conjunto original de treino tem shape (50000, 32, 32, 3).

### Quantas features cada imagem possui após o flatten?

Cada imagem passa a ter 3.072 features, porque 32 × 32 × 3 = 3.072. Cada pixel de cada canal vira uma posição no vetor de entrada.

### Por que o flatten é necessário para uma MLP?

A MLP recebe vetores unidimensionais em camadas densas. Como as imagens originalmente são matrizes com canais, o flatten transforma cada imagem em um vetor compatível com o `MLPClassifier`.

### Qual a importância da normalização para o treinamento?

A normalização coloca os pixels na escala [0, 1] em vez de [0, 255]. Isso melhora a estabilidade numérica, facilita a convergência e evita que os gradientes fiquem muito grandes durante o treinamento.


# Questão 2
Implemente a função:
```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```
## Requisitos
Sua implementação deve:
- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.
A função deve retornar o modelo treinado.
Além disso, responda:
1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?


**Solução**:


In [5]:
from sklearn.neural_network import MLPClassifier

def count_first_layer_params(n_features, hidden_layers):
    first_hidden_units = hidden_layers[0]
    return (n_features * first_hidden_units) + first_hidden_units

def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=30,
    batch_size=128,
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        solver="adam",
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=5,
        verbose=False,
    )
    model.fit(X_train, y_train)
    return model

example_layers = (64,)
print(
    "Parametros na primeira camada para arquitetura (64,):",
    count_first_layer_params(X_train.shape[1], example_layers),
)


Parametros na primeira camada para arquitetura (64,): 196672


### Quantos parâmetros existem na primeira camada?

A primeira camada possui `(número de entradas × neurônios da primeira camada) + neurônios da primeira camada`. Para o CIFAR-10 após o flatten, são 3.072 entradas. Na arquitetura `(64,)`, isso resulta em 3.072 × 64 + 64 = 196.672 parâmetros.

### Qual a função da ativação ReLU?

A ReLU introduz não linearidade na rede. Ela retorna zero para valores negativos e mantém valores positivos, ajudando a rede a aprender padrões mais complexos e reduzindo problemas de saturação em comparação com sigmoid e tanh.

### Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

Porque cada pixel de cada canal é conectado a todos os neurônios da camada seguinte. Como uma imagem 32×32×3 já gera 3.072 entradas, mesmo uma camada pequena cria muitas conexões e muitos pesos.


# Questão 3
Implemente a função:
```python
evaluate(model, X_test, y_test)
```
Ela deve:
- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.
Utilize `sklearn.metrics`.
Além disso:
- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.
Responda:
1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?


**Solução**:


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }


### O que a accuracy representa?

Accuracy representa a proporção de acertos do modelo em relação ao total de exemplos avaliados. Ela mostra quantas imagens foram classificadas corretamente no conjunto de validação.

### Qual a diferença entre precision e recall?

Precision mede, entre as previsões feitas para uma classe, quantas estavam corretas. Recall mede, entre os exemplos reais de uma classe, quantos foram encontrados pelo modelo.

### Em quais situações o f1-score é importante?

O f1-score é importante quando é necessário equilibrar precision e recall. Ele é especialmente útil em bases desbalanceadas ou quando falsos positivos e falsos negativos precisam ser analisados juntos.


# Questão 4
Implemente o rastreamento experimental utilizando MLflow.
## Devem ser registrados:
### Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size
### Métricas
- accuracy
- precision
- recall
- f1_score
- training_time
Utilize:
```python
mlflow.log_param()
mlflow.log_metric()
```
Ao final:
- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.
Responda:
1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?


**Solução**:


In [7]:
import time

mlflow.set_tracking_uri(f"file://{ROOT / 'mlruns'}")
mlflow.set_experiment("assignment-cifar10-mlp")

BATCH_SIZE = 128
MAX_ITER = 20

def run_experiment(name, activation, hidden_layers, learning_rate, seed=SEED):
    with mlflow.start_run(run_name=name):
        params = {
            "activation": activation,
            "hidden_layers": str(hidden_layers),
            "learning_rate": learning_rate,
            "max_iter": MAX_ITER,
            "batch_size": BATCH_SIZE,
            "seed": seed,
            "train_samples": len(X_train),
            "validation_samples": len(X_val),
        }
        for key, value in params.items():
            mlflow.log_param(key, value)

        start = time.time()
        model = train_mlp(
            X_train,
            y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=seed,
            max_iter=MAX_ITER,
            batch_size=BATCH_SIZE,
        )
        training_time = time.time() - start

        metrics = evaluate(model, X_val, y_val)
        metrics["training_time"] = training_time
        metrics["n_iter"] = model.n_iter_
        metrics["final_loss"] = model.loss_
        metrics["loss_std_last5"] = float(np.std(model.loss_curve_[-5:])) if len(model.loss_curve_) >= 5 else 0.0

        for key, value in metrics.items():
            mlflow.log_metric(key, float(value))

        return {
            "name": name,
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "model": model,
            **metrics,
        }

baseline = run_experiment(
    "baseline_relu_64_lr001",
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
)

pd.DataFrame([{k: v for k, v in baseline.items() if k != "model"}])


,name,activation,hidden_layers,learning_rate,accuracy,precision,recall,f1_score,training_time,n_iter,final_loss,loss_std_last5
0,baseline_relu_64_lr001,relu,"(64,)",0.001,0.310833,0.319389,0.310833,0.299391,1.490158,20,1.743555,0.016356


### Qual experimento apresentou melhor desempenho?

Considerando os experimentos executados, o melhor desempenho geral foi obtido com ativação `tanh`, arquitetura `(64,)` e `learning_rate=0.001`, com accuracy aproximada de 0,3767.

### Qual configuração apresentou maior estabilidade?

A estabilidade pode ser analisada pela variação final da loss (`loss_std_last5`). Entre as ativações testadas, a ReLU em `(64,)` com `learning_rate=0.001` apresentou menor variação final da loss, embora não tenha sido a maior accuracy.

### Qual o benefício do rastreamento experimental?

O MLflow permite registrar parâmetros, métricas e tempo de treinamento de cada execução. Isso facilita comparar modelos, reproduzir resultados e justificar qual configuração foi escolhida.


# Questão 5
Compare as funções:
- logistic
- tanh
- relu
## Requisitos
Utilize:
- mesma arquitetura;
- mesmo learning rate;
- mesma seed.
Para cada experimento:
- treine o modelo;
- avalie o modelo;
- registre no MLflow.
Depois compare:
- accuracy;
- convergência;
- estabilidade.
Responda:
1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?


**Solução**:


In [8]:
activation_results = []
for activation in ["logistic", "tanh", "relu"]:
    activation_results.append(
        run_experiment(
            f"activation_{activation}",
            activation=activation,
            hidden_layers=(64,),
            learning_rate=0.001,
        )
    )

activation_df = pd.DataFrame([{k: v for k, v in row.items() if k != "model"} for row in activation_results])
activation_df.sort_values("accuracy", ascending=False)


,name,activation,hidden_layers,learning_rate,accuracy,precision,recall,f1_score,training_time,n_iter,final_loss,loss_std_last5
1,activation_tanh,tanh,"(64,)",0.001,0.376667,0.373045,0.376667,0.368006,1.492588,20,1.592641,0.030045
0,activation_logistic,logistic,"(64,)",0.001,0.350000,0.365084,0.350000,0.330313,1.581166,17,1.634988,0.021320
2,activation_relu,relu,"(64,)",0.001,0.310833,0.319389,0.310833,0.299391,1.605926,20,1.743555,0.016356


### Qual ativação apresentou melhor convergência?

Nos resultados obtidos, a ativação `tanh` apresentou o melhor desempenho entre as ativações comparadas, com maior accuracy e maior f1-score.

### Qual ativação apresentou maior estabilidade?

A ReLU apresentou a menor variação final da loss entre as ativações testadas, indicando maior estabilidade no fim do treinamento. Porém, sua accuracy foi menor que a da `tanh` neste experimento.

### Houve diferenças significativas no treinamento?

Sim. A `tanh` atingiu accuracy aproximada de 0,3767, a `logistic` ficou em torno de 0,3500 e a `relu` ficou em torno de 0,3108. Isso mostra que a função de ativação afetou o comportamento do treinamento.

### Por que a ReLU é amplamente utilizada em Deep Learning?

A ReLU é simples, rápida e reduz problemas de saturação para valores positivos. Por isso, costuma acelerar a convergência em redes profundas e é muito usada em arquiteturas modernas.


# Questão 6
Compare as seguintes arquiteturas:
```python
(32,)
(64,)
(128, 64)
(256, 128)
```
## Requisitos
Para cada arquitetura:
- treine;
- avalie;
- registre no MLflow.
Analise:
- accuracy;
- custo computacional;
- estabilidade;
- overfitting.
Responda:
1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?


**Solução**:


In [9]:
architecture_results = []
for hidden_layers in [(32,), (64,), (128, 64), (256, 128)]:
    layer_name = "x".join(map(str, hidden_layers))
    architecture_results.append(
        run_experiment(
            f"architecture_{layer_name}",
            activation="relu",
            hidden_layers=hidden_layers,
            learning_rate=0.001,
        )
    )

architecture_df = pd.DataFrame([{k: v for k, v in row.items() if k != "model"} for row in architecture_results])
architecture_df.sort_values("accuracy", ascending=False)


,name,activation,hidden_layers,learning_rate,accuracy,precision,recall,f1_score,training_time,n_iter,final_loss,loss_std_last5
3,architecture_256x128,relu,"(256, 128)",0.001,0.361667,0.436011,0.361667,0.335139,2.699970,16,1.510512,0.034052
2,architecture_128x64,relu,"(128, 64)",0.001,0.355833,0.372397,0.355833,0.354679,2.476496,20,1.543590,0.022164
1,architecture_64,relu,"(64,)",0.001,0.310833,0.319389,0.310833,0.299391,1.562692,20,1.743555,0.016356
0,architecture_32,relu,"(32,)",0.001,0.272500,0.229402,0.272500,0.232145,1.148141,20,1.903415,0.009144


### Redes maiores sempre melhoraram os resultados?

Não necessariamente. Neste experimento, arquiteturas maiores melhoraram a accuracy em relação às menores, mas isso não é garantido em geral. Redes maiores também aumentam custo computacional e risco de overfitting.

### Qual arquitetura apresentou melhor tradeoff?

A arquitetura `(128, 64)` apresentou um bom tradeoff, pois teve desempenho próximo da `(256, 128)` com menor complexidade. A `(256, 128)` obteve a maior accuracy entre as arquiteturas, mas com mais parâmetros.

### Houve sinais de overfitting?

Há indícios possíveis de overfitting nas arquiteturas maiores, porque elas têm mais capacidade para memorizar padrões do conjunto de treino. Como a análise foi feita com validação e early stopping, o risco foi reduzido, mas ainda deve ser considerado.

### Qual arquitetura apresentou maior custo computacional?

A arquitetura `(256, 128)` apresentou o maior custo computacional, pois possui mais camadas, mais neurônios e mais parâmetros para atualizar durante o treinamento.


# Questão 7
Compare os seguintes learning rates:
```python
0.1
0.01
0.001
```
## Requisitos
Utilize:
- mesma arquitetura;
- mesma ativação;
- mesma seed.
Para cada experimento:
- treine;
- avalie;
- registre no MLflow.
Analise:
- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.
Responda:
1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?


In [10]:
learning_rate_results = []
for learning_rate in [0.1, 0.01, 0.001]:
    learning_rate_results.append(
        run_experiment(
            f"learning_rate_{learning_rate}",
            activation="relu",
            hidden_layers=(64,),
            learning_rate=learning_rate,
        )
    )

learning_rate_df = pd.DataFrame([{k: v for k, v in row.items() if k != "model"} for row in learning_rate_results])
learning_rate_df.sort_values("accuracy", ascending=False)


,name,activation,hidden_layers,learning_rate,accuracy,precision,recall,f1_score,training_time,n_iter,final_loss,loss_std_last5
2,learning_rate_0.001,relu,"(64,)",0.001,0.310833,0.319389,0.310833,0.299391,1.514710,20,1.743555,0.016356
1,learning_rate_0.01,relu,"(64,)",0.010,0.190000,0.104733,0.190000,0.108082,1.578160,20,2.042965,0.011901
0,learning_rate_0.1,relu,"(64,)",0.100,0.100000,0.010000,0.100000,0.018182,0.841442,13,2.325295,0.001290


### Qual learning rate apresentou melhor desempenho?

O learning rate `0.001` apresentou o melhor desempenho entre os valores testados, com maior accuracy e melhor f1-score.

### Qual apresentou maior instabilidade?

O learning rate `0.1` apresentou o pior comportamento. Ele levou o modelo a uma accuracy próxima de 0,10, equivalente a chute aleatório em um problema com 10 classes.

### O que acontece quando o learning rate é muito alto?

Quando o learning rate é muito alto, as atualizações dos pesos ficam grandes demais. O modelo pode ultrapassar boas regiões da função de perda, oscilar ou não convergir.

### O que acontece quando o learning rate é muito baixo?

Quando o learning rate é muito baixo, o treinamento tende a ser mais estável, mas muito lento. O modelo pode precisar de muitas iterações para chegar a uma boa solução.


# Questão 8
Com base nos experimentos realizados, escreva uma discussão contendo:
- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.
Além disso, responda:
1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?


In [11]:
all_results = pd.concat(
    [
        pd.DataFrame([{k: v for k, v in baseline.items() if k != "model"}]),
        activation_df,
        architecture_df,
        learning_rate_df,
    ],
    ignore_index=True,
)

summary = all_results.sort_values("accuracy", ascending=False).reset_index(drop=True)
summary


,name,activation,hidden_layers,learning_rate,accuracy,precision,recall,f1_score,training_time,n_iter,final_loss,loss_std_last5
0,activation_tanh,tanh,"(64,)",0.001,0.376667,0.373045,0.376667,0.368006,1.492588,20,1.592641,0.030045
1,architecture_256x128,relu,"(256, 128)",0.001,0.361667,0.436011,0.361667,0.335139,2.699970,16,1.510512,0.034052
2,architecture_128x64,relu,"(128, 64)",0.001,0.355833,0.372397,0.355833,0.354679,2.476496,20,1.543590,0.022164
3,activation_logistic,logistic,"(64,)",0.001,0.350000,0.365084,0.350000,0.330313,1.581166,17,1.634988,0.021320
4,baseline_relu_64_lr001,relu,"(64,)",0.001,0.310833,0.319389,0.310833,0.299391,1.490158,20,1.743555,0.016356
5,activation_relu,relu,"(64,)",0.001,0.310833,0.319389,0.310833,0.299391,1.605926,20,1.743555,0.016356
6,architecture_64,relu,"(64,)",0.001,0.310833,0.319389,0.310833,0.299391,1.562692,20,1.743555,0.016356
7,learning_rate_0.001,relu,"(64,)",0.001,0.310833,0.319389,0.310833,0.299391,1.514710,20,1.743555,0.016356
8,architecture_32,relu,"(32,)",0.001,0.272500,0.229402,0.272500,0.232145,1.148141,20,1.903415,0.009144
9,learning_rate_0.01,relu,"(64,)",0.010,0.190000,0.104733,0.190000,0.108082,1.578160,20,2.042965,0.011901


### Comportamento geral dos experimentos

A loss melhorou quando os hiperparâmetros ficaram em uma faixa mais adequada, principalmente com `learning_rate=0.001`. Valores maiores, como `0.1`, prejudicaram a convergência e deixaram o modelo praticamente no nível de chute aleatório.

A arquitetura também influenciou os resultados. Redes maiores aumentaram a capacidade de aprendizado, mas também aumentaram o número de parâmetros e o custo computacional. A função de ativação teve impacto claro: neste recorte, `tanh` teve melhor desempenho geral, enquanto ReLU apresentou boa estabilidade de loss.

A principal limitação da MLP para imagens é que o flatten remove a estrutura espacial. A rede deixa de enxergar relações locais entre pixels vizinhos, algo que CNNs exploram melhor por meio de filtros convolucionais.

### Qual configuração apresentou melhor resultado final?

A melhor configuração final foi ativação `tanh`, arquitetura `(64,)` e `learning_rate=0.001`, com accuracy aproximada de 0,3767 e f1-score aproximado de 0,3680.

### Quais foram as principais dificuldades observadas?

As principais dificuldades foram a sensibilidade ao learning rate, o custo de treinar redes densas com muitas entradas e a perda de informação espacial causada pelo flatten das imagens.

### Por que MLPs possuem limitações para imagens?

MLPs tratam cada pixel como uma feature independente. Elas não aproveitam naturalmente vizinhança espacial, bordas, texturas e padrões locais, o que limita sua eficiência em problemas de visão computacional.

### Como o backpropagation contribui para o aprendizado da rede?

O backpropagation calcula como cada peso contribuiu para o erro final. Com esses gradientes, o otimizador ajusta os pesos camada por camada para reduzir a loss e melhorar as predições.
